# Ingestão — raw, bronze e quarentena

Baixa as três fontes do projeto (EPE, NOAA ONI, Open-Meteo) e grava cada execução como
uma **carga** identificada por `ID_CARGA`. Nada é sobrescrito: a carga nova vai ao lado
das anteriores.

| Camada | Caminho | Conteúdo |
| --- | --- | --- |
| raw | `data/raw/<fonte>/<id_carga>/` | bytes exatamente como vieram + sidecar `.meta.json` |
| bronze | `data/bronze/<fonte>/<id_carga>.parquet` | um registro da origem por linha, colunas originais como `string` + metadados técnicos |
| quarentena | `data/quarentena/<fonte>/<id_carga>.parquet` | registros que falharam no parse estrutural, com o motivo |

Metadados técnicos de cada registro na bronze: `_id_carga`, `_extraido_em`, `_origem`
(URL), `_arquivo_raw`, `_linha` e `_hash_registro` (sha256 do registro bruto).

A quarentena pega só erro **estrutural** (encoding, número de campos, campo obrigatório
vazio). Regra de negócio — faixa, sentinela `-99.9`, calendário — é papel da silver.
As armadilhas de cada fonte estão em `docs/dataset.md`.

Rode de cima para baixo com `notebooks/` como diretório de trabalho.

In [ ]:
import hashlib
import io
import json
import time
from datetime import datetime, timezone
from pathlib import Path

import pandas as pd
import requests

RAIZ = Path.cwd().parent
DATA_DIR = RAIZ / "data"
HEADERS = {"User-Agent": "CESUPA-ETL-Lab/1.0"}

START_YEAR = 2004
END_YEAR = 2025

ID_CARGA = datetime.now(timezone.utc).strftime("%Y%m%dT%H%M%SZ")
print(f"carga {ID_CARGA}")

## Helpers

In [ ]:
def baixar(url, params=None, tentativas=6, espera_base=1.6, timeout=60):
    """GET com timeout e retry exponencial. 429/5xx é transitório; outro 4xx é definitivo."""
    for tentativa in range(1, tentativas + 1):
        try:
            resposta = requests.get(url, params=params, headers=HEADERS, timeout=timeout)
        except requests.RequestException as erro:
            if tentativa == tentativas:
                raise
            print(f"  rede falhou ({type(erro).__name__}); tentativa {tentativa}/{tentativas}")
            time.sleep(espera_base ** tentativa)
            continue

        if resposta.status_code == 200:
            return resposta

        if resposta.status_code in (429, 500, 502, 503, 504):
            # O servidor sabe melhor quanto esperar; o backoff é só o plano B.
            retry_after = resposta.headers.get("Retry-After", "")
            espera = float(retry_after) if retry_after.isdigit() else espera_base ** tentativa
            print(f"  status {resposta.status_code}; aguardando {espera:.1f}s "
                  f"(tentativa {tentativa}/{tentativas})")
            time.sleep(espera)
            continue

        raise RuntimeError(f"falha definitiva {resposta.status_code} em {resposta.url}: "
                           f"{resposta.text[:200]}")

    raise RuntimeError(f"desisti de {url} após {tentativas} tentativas")


def salvar_raw(conteudo, fonte, nome, origem, **extra):
    """Grava os bytes crus da carga e, ao lado, o sidecar. Devolve o sidecar."""
    destino = DATA_DIR / "raw" / fonte / ID_CARGA / nome
    destino.parent.mkdir(parents=True, exist_ok=True)
    destino.write_bytes(conteudo)

    meta = {
        "_fonte": fonte,
        "_origem": origem,
        "_arquivo_raw": str(destino.relative_to(RAIZ)),
        "_extraido_em": datetime.now(timezone.utc).isoformat(timespec="seconds"),
        "_id_carga": ID_CARGA,
        "_sha256": hashlib.sha256(conteudo).hexdigest(),
        **extra,
    }
    destino.with_suffix(destino.suffix + ".meta.json").write_text(
        json.dumps(meta, ensure_ascii=False, indent=2), encoding="utf-8")
    return meta


def registro(meta, linha, bruto, campos):
    """Um registro da bronze: campos da origem + rastreio. `_bruto` vira o hash."""
    return {**campos, "_origem": meta["_origem"], "_arquivo_raw": meta["_arquivo_raw"],
            "_extraido_em": meta["_extraido_em"], "_linha": linha, "_bruto": bruto}


def rejeitado(meta, linha, bruto, motivo):
    """Um registro da quarentena: o que veio, de onde e por que não entrou."""
    return {"_origem": meta["_origem"], "_arquivo_raw": meta["_arquivo_raw"],
            "_extraido_em": meta["_extraido_em"], "_linha": linha,
            "_registro_bruto": bruto, "_motivo": motivo}


def ler_delimitado(conteudo, meta, colunas, encoding, sep=None, primeira_linha=1):
    """Bytes -> (registros, quarentena). Decodifica linha a linha no encoding declarado,
    para que um byte podre derrube só a linha dele, e quebra no separador
    (sep=None = espaço em branco)."""
    registros, quarentena = [], []
    for n, bruta in enumerate(conteudo.splitlines(), start=primeira_linha):
        try:
            texto = bruta.decode(encoding)
        except UnicodeDecodeError as erro:
            quarentena.append(rejeitado(meta, n, bruta.decode(encoding, "replace"),
                                        f"encoding {encoding}: {erro.reason}"))
            continue

        if not texto.strip():
            continue

        campos = texto.split(sep)
        if len(campos) != len(colunas):
            quarentena.append(rejeitado(meta, n, texto,
                                        f"n_campos: esperado {len(colunas)}, veio {len(campos)}"))
            continue

        registros.append(registro(meta, n, texto, dict(zip(colunas, campos))))
    return registros, quarentena


COLUNAS_QUARENTENA = ["_origem", "_arquivo_raw", "_extraido_em", "_linha",
                      "_registro_bruto", "_motivo"]


def gravar_bronze(fonte, registros, quarentena):
    """Grava bronze e quarentena da carga, tudo como string."""
    assert registros, f"{fonte}: nenhum registro válido — a carga inteira caiu na quarentena?"

    bronze = pd.DataFrame(registros)
    bronze["_hash_registro"] = [hashlib.sha256(b.encode("utf-8")).hexdigest()
                                for b in bronze.pop("_bruto")]
    rejeitados = pd.DataFrame(quarentena, columns=COLUNAS_QUARENTENA)

    for camada, df in (("bronze", bronze), ("quarentena", rejeitados)):
        df["_id_carga"] = ID_CARGA
        destino = DATA_DIR / camada / fonte / f"{ID_CARGA}.parquet"
        destino.parent.mkdir(parents=True, exist_ok=True)
        df.astype("string").to_parquet(destino, index=False)

    print(f"  {fonte}: {len(bronze)} na bronze, {len(rejeitados)} na quarentena")

### Checagem da quarentena

Bytes sintéticos com uma linha boa, uma com campo a mais e uma com um byte que não é
utf-8. Só a boa pode chegar à bronze.

In [ ]:
def testar_quarentena():
    meta = {"_origem": "teste", "_arquivo_raw": "teste", "_extraido_em": "agora"}
    conteudo = b"2015;1,5\n2016;2,5;extra\n\xff2017;3,5\n"

    registros, quarentena = ler_delimitado(conteudo, meta, ["ano", "valor"], "utf-8", sep=";")

    assert [r["ano"] for r in registros] == ["2015"], registros
    assert [q["_linha"] for q in quarentena] == [2, 3], quarentena
    assert quarentena[0]["_motivo"].startswith("n_campos")
    assert quarentena[1]["_motivo"].startswith("encoding")
    print("quarentena OK — 1 registro aceito, 2 rejeitados com o motivo certo")


testar_quarentena()

## 1. EPE — consumo mensal de energia elétrica (target)

Planilha única com o histórico inteiro (2004 em diante); o recorte da janela é da silver.
Duas abas viram duas tabelas bronze: por subsistema e por UF.

In [ ]:
URL_EPE = ("https://www.epe.gov.br/sites-pt/publicacoes-dados-abertos/dados-abertos/"
           "Documents/Dados_abertos_Consumo_Mensal.xlsx")

ABAS_EPE = {
    "epe_consumo_subsistema": "CONSUMO E NUMCONS SAM",
    "epe_consumo_uf": "CONSUMO E NUMCONS SAM UF",
}
OBRIGATORIAS_EPE = ["Data", "Classe", "Consumo"]

conteudo = baixar(URL_EPE).content
meta_epe = salvar_raw(conteudo, "epe_consumo", "consumo_mensal_historico.xlsx", URL_EPE)

for fonte, aba in ABAS_EPE.items():
    planilha = pd.read_excel(io.BytesIO(conteudo), sheet_name=aba, dtype=str)
    meta = {**meta_epe, "_origem": f"{URL_EPE}#{aba}"}
    registros, quarentena = [], []

    for indice, linha in planilha.iterrows():
        campos = {c: (None if pd.isna(v) else v) for c, v in linha.items()}
        bruto = json.dumps(campos, ensure_ascii=False, sort_keys=True)
        n = indice + 2  # linha do Excel: o cabeçalho ocupa a 1

        vazias = [c for c in OBRIGATORIAS_EPE if campos.get(c) is None]
        if vazias:
            quarentena.append(rejeitado(meta, n, bruto, f"obrigatória vazia: {vazias}"))
        else:
            registros.append(registro(meta, n, bruto, campos))

    gravar_bronze(fonte, registros, quarentena)

## 2. NOAA — Oceanic Niño Index (sinal climático)

Texto de largura fixa, separado por espaço, utf-8 (o rodapé tem "Niño"). A primeira linha
declara o primeiro e o último ano; o bloco de dados é exatamente esse intervalo. O rodapé
(sentinela `-99.9` e versão do ERSST) não é registro: vai para o sidecar.

In [ ]:
URL_ONI = "https://psl.noaa.gov/data/correlation/oni.data"
COLUNAS_ONI = ["ano", *[f"m{mes:02d}" for mes in range(1, 13)]]

conteudo = baixar(URL_ONI).content
linhas = conteudo.splitlines()
primeiro_ano, ultimo_ano = map(int, linhas[0].split())
n_anos = ultimo_ano - primeiro_ano + 1

meta_oni = salvar_raw(conteudo, "noaa_oni", "oni.data", URL_ONI,
                      _anos_declarados=[primeiro_ano, ultimo_ano],
                      _rodape=[l.decode("utf-8").strip() for l in linhas[1 + n_anos:] if l.strip()])

registros, quarentena = ler_delimitado(b"\n".join(linhas[1:1 + n_anos]), meta_oni,
                                       COLUNAS_ONI, "utf-8", primeira_linha=2)

# Estrutural: 13 campos que não são números também é linha podre.
def numerica(r):
    try:
        [float(r[c]) for c in COLUNAS_ONI]
        return True
    except ValueError:
        return False

quarentena += [rejeitado(meta_oni, r["_linha"], r["_bruto"], "campo não numérico")
               for r in registros if not numerica(r)]
registros = [r for r in registros if numerica(r)]

gravar_bronze("noaa_oni", registros, quarentena)

## 3. Open-Meteo — clima diário por capital de UF (API REST paginada)

Uma capital por UF; a silver decide como agregar por subsistema. A API não pagina por
cursor, então a paginação é **por janela de tempo**: uma página por UF × ano. Cada
página é um arquivo raw; cada dia vira um registro, com a latitude/longitude da célula de
grade que a API devolveu (não o ponto pedido — as duas ficam no sidecar).

In [ ]:
URL_CLIMA = "https://archive-api.open-meteo.com/v1/archive"
VARIAVEIS = ["temperature_2m_mean", "temperature_2m_max", "precipitation_sum"]

CAPITAIS = {
    "AC": (-9.97, -67.81),   # Rio Branco
    "AL": (-9.67, -35.74),   # Maceió
    "AP": (0.03, -51.07),    # Macapá
    "AM": (-3.12, -60.02),   # Manaus
    "BA": (-12.97, -38.50),  # Salvador
    "CE": (-3.73, -38.53),   # Fortaleza
    "DF": (-15.78, -47.93),  # Brasília
    "ES": (-20.32, -40.34),  # Vitória
    "GO": (-16.68, -49.25),  # Goiânia
    "MA": (-2.53, -44.30),   # São Luís
    "MT": (-15.60, -56.10),  # Cuiabá
    "MS": (-20.47, -54.62),  # Campo Grande
    "MG": (-19.92, -43.94),  # Belo Horizonte
    "PA": (-1.46, -48.49),   # Belém
    "PB": (-7.12, -34.86),   # João Pessoa
    "PR": (-25.43, -49.27),  # Curitiba
    "PE": (-8.05, -34.88),   # Recife
    "PI": (-5.09, -42.80),   # Teresina
    "RJ": (-22.91, -43.17),  # Rio de Janeiro
    "RN": (-5.79, -35.21),   # Natal
    "RS": (-30.03, -51.23),  # Porto Alegre
    "RO": (-8.76, -63.90),   # Porto Velho
    "RR": (2.82, -60.67),    # Boa Vista
    "SC": (-27.59, -48.55),  # Florianópolis
    "SP": (-23.55, -46.63),  # São Paulo
    "SE": (-10.91, -37.07),  # Aracaju
    "TO": (-10.18, -48.33),  # Palmas
}


def paginar_por_ano(url, params):
    """Uma requisição por ano da janela. Devolve (ano, resposta) página a página."""
    for ano in range(START_YEAR, END_YEAR + 1):
        pagina = {**params, "start_date": f"{ano}-01-01", "end_date": f"{ano}-12-31"}
        yield ano, baixar(url, pagina)
        # ponytail: pausa fixa. Uma página (1 ano × 3 variáveis) pesa ~8 no limite de
        # 600/min do plano gratuito; 1 s dá folga. Mudou janela ou variáveis? Refaça a conta.
        # Conta 2004–2025: 27 UFs × 22 anos = 594 páginas ≈ 4.600 de peso, perto do limite
        # de 5.000/h. Rode uma vez só; se der 429, espere a hora virar antes de repetir.
        time.sleep(1)


registros, quarentena = [], []
for uf, (lat, lon) in CAPITAIS.items():
    params = {"latitude": lat, "longitude": lon, "daily": ",".join(VARIAVEIS),
              "timezone": "America/Sao_Paulo"}

    for ano, resposta in paginar_por_ano(URL_CLIMA, params):
        corpo = resposta.json()
        meta = salvar_raw(resposta.content, "openmeteo_clima", f"clima_{uf}_{ano}.json",
                          resposta.url,
                          _coordenada_pedida=[lat, lon],
                          _coordenada_grade=[corpo["latitude"], corpo["longitude"]],
                          _unidades=corpo.get("daily_units"))

        diario = corpo.get("daily", {})
        tamanhos = {c: len(diario.get(c) or []) for c in ["time", *VARIAVEIS]}
        if len(set(tamanhos.values())) != 1 or not tamanhos["time"]:
            quarentena.append(rejeitado(meta, None, resposta.text,
                                        f"arrays desalinhados ou vazios: {tamanhos}"))
            continue

        for i, dia in enumerate(diario["time"]):
            campos = {"latitude": corpo["latitude"], "longitude": corpo["longitude"],
                      "time": dia, **{c: diario[c][i] for c in VARIAVEIS}}
            bruto = json.dumps(campos, sort_keys=True)
            campos = {c: (None if v is None else str(v)) for c, v in campos.items()}
            registros.append(registro(meta, i, bruto, campos))

    print(f"  {uf}: {END_YEAR - START_YEAR + 1} páginas")

gravar_bronze("openmeteo_clima", registros, quarentena)

## Conferência

Para cada fonte desta carga: todo raw tem sidecar, a bronze tem os metadados técnicos, o
hash não repete e a contagem bate com o que a origem declara.

In [ ]:
METADADOS = ["_id_carga", "_extraido_em", "_origem", "_arquivo_raw", "_linha", "_hash_registro"]
dias = len(pd.date_range(f"{START_YEAR}-01-01", f"{END_YEAR}-12-31"))

ESPERADO = {
    "epe_consumo_subsistema": None,
    "epe_consumo_uf": None,
    "noaa_oni": n_anos,
    "openmeteo_clima": len(CAPITAIS) * dias,
}

for pasta in ["epe_consumo", "noaa_oni", "openmeteo_clima"]:
    for arquivo in (DATA_DIR / "raw" / pasta / ID_CARGA).iterdir():
        if not arquivo.name.endswith(".meta.json"):
            assert arquivo.with_suffix(arquivo.suffix + ".meta.json").exists(), f"sem sidecar: {arquivo}"

for fonte, esperado in ESPERADO.items():
    bronze = pd.read_parquet(DATA_DIR / "bronze" / fonte / f"{ID_CARGA}.parquet")
    rejeitados = pd.read_parquet(DATA_DIR / "quarentena" / fonte / f"{ID_CARGA}.parquet")

    faltando = set(METADADOS) - set(bronze.columns)
    assert not faltando, f"{fonte}: sem metadados {faltando}"
    assert not bronze["_hash_registro"].duplicated().any(), f"{fonte}: hash repetido na carga"
    if esperado is not None:
        assert len(bronze) + len(rejeitados) == esperado, \
            f"{fonte}: {len(bronze)} + {len(rejeitados)} != {esperado}"

    print(f"{fonte:<24} bronze {len(bronze):>7}   quarentena {len(rejeitados):>4}")

print(f"\ncarga {ID_CARGA} OK")